# Advanced Processing Options

This notebook covers advanced features: PMTiles generation, STAC customization, deduplication, and direct processor usage.

In [ ]:
# Setup
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

## PMTiles Generation

PMTiles are cloud-optimized vector tiles for web map visualization:

In [ ]:
from oceanstream import convert
import tempfile

input_dir = project_root / "oceanstream" / "tests" / "data" / "raw_data"
output_dir = Path(tempfile.mkdtemp()) / "pmtiles_demo"

# Generate PMTiles alongside GeoParquet
convert(
    provider="saildrone",
    input_source=input_dir,
    output_dir=output_dir,
    campaign_id="pmtiles_demo",
    generate_pmtiles=True,       # Enable PMTiles generation
    pmtiles_minzoom=0,           # Minimum zoom level
    pmtiles_maxzoom=12,          # Maximum zoom level
    pmtiles_sample_rate=10,      # Sample every Nth point
    pmtiles_include_measurements=True,  # Include sensor data
    verbose=True,
    yes=True,
)

# PMTiles will be at: output_dir/campaign_id/tiles/track.pmtiles

## Deduplication Control

OceanStream tracks processed files to avoid reprocessing:

In [ ]:
# Force reprocess (ignore previous processing)
convert(
    provider="saildrone",
    input_source=input_dir,
    output_dir=output_dir,
    campaign_id="dedup_demo",
    force_reprocess=True,  # Reprocess all files
    yes=True,
)

In [ ]:
# Allow duplicate rows (skip row-level deduplication)
convert(
    provider="saildrone",
    input_source=input_dir,
    output_dir=output_dir,
    campaign_id="no_dedup_demo",
    deduplicate=False,  # Keep all rows even if duplicates
    yes=True,
)

## Dry Run Mode

Preview what would be processed without writing files:

In [ ]:
# Dry run - shows files that would be processed
result = convert(
    provider="saildrone",
    input_source=input_dir,
    output_dir=output_dir,
    campaign_id="dryrun_demo",
    dry_run=True,  # No files written
    verbose=True,
    yes=True,
)

## Using GeotrackProcessor Directly

For fine-grained control, use the processor class directly:

In [ ]:
from oceanstream.geotrack.processor import GeotrackProcessor
from oceanstream import get_provider

# Create processor
processor = GeotrackProcessor(
    provider=get_provider("saildrone"),
    output_dir=output_dir,
    generate_pmtiles=False,
    bin_size=1.0,  # 1° lat/lon bins (default)
)

# Process a single file
csv_file = input_dir / "sd1033_tpos_2023_gps_1min.csv"
if csv_file.exists():
    result = processor.process_file(
        file_path=csv_file,
        campaign_id="direct_processor",
    )
    print(f"Processed: {result}")

## Custom Metadata Attribution

Add custom attribution to STAC metadata:

In [ ]:
convert(
    provider="saildrone",
    input_source=input_dir,
    output_dir=output_dir,
    campaign_id="custom_meta",
    stac_title="My Custom Campaign Title",
    stac_description="Detailed description of the data collection",
    stac_license="CC-BY-4.0",
    verbose=True,
    yes=True,
)